# Demo thiết kế CSDL — Show, don't tell

Notebook này dành cho buổi demo với giảng viên hướng dẫn. Mạch trình bày:

1. Bắt đầu bằng **một giao dịch thật** để người nghe hiểu câu chuyện bán hàng.
2. Cho thấy **khóa sai tạo kết quả sai**.
3. Cho thấy **dữ liệu lặp** trước khi nói tới 1NF/2NF/3NF.
4. Dùng dữ liệu thật để giải thích quyết định tách geography và bảng `region`.
5. Tái tạo `sales.csv` để chứng minh vì sao nó nên là view.

> Cách demo: chạy từng cell, đọc kết quả rồi mới kết luận. Không bắt đầu bằng việc đọc 19 bảng trên ERD.

In [1]:
import pandas as pd, numpy as np
import warnings; warnings.filterwarnings('ignore')
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 60)

DATA = 'data'
R = lambda f, **k: pd.read_csv(f'{DATA}/{f}', low_memory=False, **k)

geo    = R('geography.csv')
cust   = R('customers.csv', parse_dates=['signup_date'])
orders = R('orders.csv', parse_dates=['order_date'])
items  = R('order_items.csv')
prod   = R('products.csv')
promo  = R('promotions.csv', parse_dates=['start_date', 'end_date'])
pay    = R('payments.csv')
ret    = R('returns.csv', parse_dates=['return_date'])
rev    = R('reviews.csv', parse_dates=['review_date'])
inv    = R('inventory.csv', parse_dates=['snapshot_date'])
sales  = R('sales.csv', parse_dates=['Date'])

items = items.assign(line_number=items.groupby('order_id', sort=False).cumcount() + 1)
print(f'Đã nạp {sum(map(len, [geo,cust,orders,items,prod,promo,pay,ret,rev,inv,sales])):,} dòng từ 11 file cần cho demo.')

Đã nạp 2,390,469 dòng từ 11 file cần cho demo.


## Demo 1 — Một đơn hàng thật đi qua những bảng nào?

**Câu nói:** “Trước khi nói PK/FK, em lần theo một giao dịch thật. Một order là phần đầu đơn; một order item là một dòng sản phẩm trong đơn.”

In [2]:
order_id = 46270

order_demo = (orders.query('order_id == @order_id').drop(columns='zip')
              .merge(cust, on='customer_id', validate='many_to_one')
              .merge(geo, on='zip', suffixes=('_customer', '_geo'), validate='many_to_one'))
item_demo = (items.query('order_id == @order_id')
             .merge(prod, on='product_id', validate='many_to_one'))
payment_demo = pay.query('order_id == @order_id')

display(order_demo[['order_id','order_date','customer_id','zip','city_geo','district','region',
                    'order_status','payment_method','device_type','order_source']])
display(item_demo[['order_id','line_number','product_id','product_name','category','quantity',
                   'unit_price','discount_amount','promo_id','promo_id_2']])
display(payment_demo)

gross = (item_demo.quantity * item_demo.unit_price).sum()
net = gross - item_demo.discount_amount.sum()
print(f'Gross Revenue của đơn = {gross:,.2f}')
print(f'Net Payment tính từ dòng hàng = {net:,.2f}')
print(f'payment_value trong file = {payment_demo.payment_value.iloc[0]:,.2f}')

,order_id,order_date,customer_id,zip,city_geo,district,region,order_status,payment_method,device_type,order_source
0,46270,2013-02-04,44051,21122,Bac Ninh,District #05,East,delivered,paypal,mobile,social_media


,order_id,line_number,product_id,product_name,category,quantity,unit_price,discount_amount,promo_id,promo_id_2
0,46270,1,2254,VietMotion RP-51,Outdoor,4,615.65,369.39,PROMO-0006,NaN


,order_id,payment_method,payment_value,installments
35810,46270,paypal,2093.21,3


Gross Revenue của đơn = 2,462.60
Net Payment tính từ dòng hàng = 2,093.21
payment_value trong file = 2,093.21


### Chốt sau Demo 1

- `orders`: một dòng = một đơn, khóa `order_id`.
- `order_items`: một dòng = một mặt hàng trong đơn; cần `(order_id, line_number)`.
- `payments`: phần thanh toán của đơn, nối bằng `order_id`.
- Câu chuyện: khách ở một nơi → đặt đơn → đơn có dòng hàng → thanh toán.

## Demo 2 — Vì sao không dùng `(order_id, product_id)` làm khóa dòng hàng?

**Câu nói:** “Em không tự chọn khóa theo cảm giác; em thử tính duy nhất và thử hậu quả khi join.”

In [3]:
dup = items[items.duplicated(['order_id','product_id'], keep=False)].sort_values(['order_id','product_id'])
print(f'{len(dup)} dòng thuộc {dup.order_id.nunique()} đơn có (order_id, product_id) trùng.')
display(dup.head(8))

wrong_join = ret.merge(items[['order_id','product_id','quantity']],
                       on=['order_id','product_id'], how='left')

item_occurrence = items.copy()
item_occurrence['_occurrence'] = item_occurrence.groupby(['order_id','product_id'], sort=False).cumcount()
return_occurrence = ret.copy()
return_occurrence['_occurrence'] = return_occurrence.groupby(['order_id','product_id'], sort=False).cumcount()
right_join = return_occurrence.merge(
    item_occurrence[['order_id','product_id','_occurrence','line_number','quantity']],
    on=['order_id','product_id','_occurrence'], how='left', validate='one_to_one')

comparison = pd.DataFrame({
    'Cách nối': ['Sai: order_id + product_id', 'Đúng sau khi gán line_number'],
    'Số dòng kết quả': [len(wrong_join), len(right_join)],
    'Fan-out': [len(wrong_join)-len(ret), len(right_join)-len(ret)],
    'return_quantity > quantity': [
        (wrong_join.return_quantity > wrong_join.quantity).sum(),
        (right_join.return_quantity > right_join.quantity).sum()]
})
display(comparison)

32 dòng thuộc 16 đơn có (order_id, product_id) trùng.


,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2,line_number
12233,14280,976,1,4019.47,0.0,NaN,NaN,1
12234,14280,976,2,3937.99,0.0,NaN,NaN,2
99645,113379,786,6,694.34,0.0,NaN,NaN,1
99646,113379,786,1,699.37,0.0,NaN,NaN,2
189239,215525,1859,5,1896.11,0.0,NaN,NaN,1
189240,215525,1859,5,1897.20,0.0,NaN,NaN,2
190291,216740,791,8,793.10,0.0,NaN,NaN,1
190292,216740,791,5,825.52,0.0,NaN,NaN,2


,Cách nối,Số dòng kết quả,Fan-out,return_quantity > quantity
0,Sai: order_id + product_id,39943,4,1
1,Đúng sau khi gán line_number,39939,0,0


### Chốt sau Demo 2

Khóa sai không chỉ “kém đẹp”: nó làm nở số dòng và tạo lỗi nghiệp vụ giả. Vì nguồn không có khóa dòng hàng, ETL phải sinh `line_number` ổn định trước khi nối `returns` hoặc `reviews`.

## Demo 3 — Cho thấy dữ liệu lặp rồi mới nói chuẩn hóa

**Câu nói:** “Mỗi sự thật nên có một nơi sở hữu. Em kiểm chứng chỗ nào đang lưu cùng một sự thật nhiều lần.”

In [4]:
customer_city = cust[['customer_id','zip','city']].merge(
    geo[['zip','city']], on='zip', suffixes=('_customer','_geography'))
order_zip = orders[['order_id','customer_id','zip']].merge(
    cust[['customer_id','zip']], on='customer_id', suffixes=('_order','_customer'))

promo_long = (items[['order_id','line_number','promo_id','promo_id_2']]
              .melt(id_vars=['order_id','line_number'], value_name='promo_code')
              .dropna(subset=['promo_code']))

evidence = pd.DataFrame([
    ('customers.city so với geography.city',
     (customer_city.city_customer != customer_city.city_geography).sum(),
     'Bỏ customers.city; suy qua zip'),
    ('orders.zip so với customers.zip',
     (order_zip.zip_order != order_zip.zip_customer).sum(),
     'Bỏ orders.zip; suy qua customer_id'),
    ('Dòng có promo_id_2', items.promo_id_2.notna().sum(),
     'Unpivot thành order_item_promotion'),
    ('Số liên kết item-promotion sau unpivot', len(promo_long),
     'Một promo mới chỉ thêm dòng, không thêm cột')
], columns=['Bằng chứng', 'Số trường hợp/lệch', 'Quyết định thiết kế'])
display(evidence)

,Bằng chứng,Số trường hợp/lệch,Quyết định thiết kế
0,customers.city so với geography.city,0,Bỏ customers.city; suy qua zip
1,orders.zip so với customers.zip,0,Bỏ orders.zip; suy qua customer_id
2,Dòng có promo_id_2,206,Unpivot thành order_item_promotion
3,Số liên kết item-promotion sau unpivot,276522,"Một promo mới chỉ thêm dòng, không thêm cột"


## Demo 4 — Geography: dữ liệu thật nói gì?

**Câu nói:** “Em kiểm tra phụ thuộc hàm trước, sau đó mới chọn cấu trúc bảng. Đây là phần có đánh đổi, không phải đáp án duy nhất.”

In [5]:
def fd_result(df, determinant, dependent):
    counts = df.groupby(determinant)[dependent].nunique()
    return int((counts > 1).sum()), int(counts.max())

fd_rows = []
for determinant, dependent in [
    ('zip','city'), ('zip','district'), ('city','region'),
    ('district','region'), ('city','district'), ('district','city')]:
    violations, max_values = fd_result(geo, determinant, dependent)
    fd_rows.append((f'{determinant} → {dependent}', violations, max_values,
                    'ĐÚNG' if violations == 0 else 'SAI'))

display(pd.DataFrame(fd_rows, columns=['Phụ thuộc hàm','Số determinant vi phạm',
                                        'Số dependent tối đa','Kết luận']))
print(f"Quy mô: {geo.zip.nunique():,} zip, {geo.city.nunique()} city, "
      f"{geo.district.nunique()} district, {geo.region.nunique()} region.")

,Phụ thuộc hàm,Số determinant vi phạm,Số dependent tối đa,Kết luận
0,zip → city,0,1,ĐÚNG
1,zip → district,0,1,ĐÚNG
2,city → region,0,1,ĐÚNG
3,district → region,0,1,ĐÚNG
4,city → district,42,19,SAI
5,district → city,39,16,SAI


Quy mô: 39,948 zip, 42 city, 39 district, 3 region.


### Điều kết quả vừa chứng minh

- `zip → city`, `zip → district`, `city → region`, `district → region` đều đúng trong dữ liệu.
- Nhưng `city → district` và `district → city` đều sai: hai chiều này cắt chéo, không tạo thành một cây địa lý đơn giản.
- Tách `city`, `district`, `zip_area` giảm lặp ở grain zip. Tuy nhiên region có thể đi từ zip theo hai đường, nên cần ETL test để hai đường khớp nhau.

## Demo 5 — Bảng `region` một cột thực sự làm được gì?

**Câu nói:** “Nếu bỏ bảng `region` nhưng giữ cột region trong city và district thì không mất thông tin. Bảng một cột này chỉ có giá trị hiện tại như một danh mục kiểm soát miền bằng FK.”

In [6]:
region_table = geo[['region']].drop_duplicates().sort_values('region').reset_index(drop=True)
city_table = geo[['city','region']].drop_duplicates().sort_values('city').reset_index(drop=True)
district_table = geo[['district','region']].drop_duplicates().sort_values('district').reset_index(drop=True)

print('Danh mục region dùng làm đích FK:')
display(region_table)

incoming_city = pd.DataFrame({'city': ['City hợp lệ', 'City nhập sai'],
                              'region': [region_table.region.iloc[0], 'Eats']})
incoming_city['FK_region_hợp_lệ'] = incoming_city.region.isin(region_table.region)
display(incoming_city)

pairs_before = len(city_table) + len(district_table)
pairs_after_removing_region_master = len(city_table) + len(district_table)
print(f'Số quan hệ city/district → region trước khi bỏ bảng master: {pairs_before}')
print(f'Số quan hệ còn lại sau khi bỏ bảng master: {pairs_after_removing_region_master}')
print('Mất thông tin: 0. Mất khả năng FK kiểm soát miền: có, nếu không thay bằng CHECK/ENUM.')

Danh mục region dùng làm đích FK:


,region
0,Central
1,East
2,West


,city,region,FK_region_hợp_lệ
0,City hợp lệ,Central,True
1,City nhập sai,Eats,False


Số quan hệ city/district → region trước khi bỏ bảng master: 81
Số quan hệ còn lại sau khi bỏ bảng master: 81
Mất thông tin: 0. Mất khả năng FK kiểm soát miền: có, nếu không thay bằng CHECK/ENUM.


### Chốt sau Demo 5

- Không có bảng `region` vẫn có thể đạt 3NF: `city(city PK, region)` và `district(district PK, region)`.
- Giữ bảng `region` khi muốn quản lý master data, dùng FK, hoặc dự kiến region có thêm `region_id`, `region_name`, `manager`, ...
- Nếu region chỉ là ba nhãn cố định, `CHECK`/`ENUM` là phương án đơn giản hơn. Đây là **judgment call**, không phải 3NF bắt buộc.

## Demo 6 — Vì sao `sales.csv` là view, không phải bảng cơ sở?

**Câu nói:** “Em không chỉ nói sales là dữ liệu dẫn xuất; em tái tạo toàn bộ 3.833 ngày từ các dòng giao dịch rồi đo sai số.”

In [7]:
sales_rebuilt = (items
    .merge(orders[['order_id','order_date']], on='order_id', validate='many_to_one')
    .merge(prod[['product_id','cogs']], on='product_id', validate='many_to_one'))
sales_rebuilt['Revenue_rebuilt'] = sales_rebuilt.quantity * sales_rebuilt.unit_price
sales_rebuilt['COGS_rebuilt'] = sales_rebuilt.quantity * sales_rebuilt.cogs
daily_rebuilt = (sales_rebuilt.groupby('order_date', as_index=False)
                 .agg(Revenue_rebuilt=('Revenue_rebuilt','sum'),
                      COGS_rebuilt=('COGS_rebuilt','sum')))
check = sales.merge(daily_rebuilt, left_on='Date', right_on='order_date', validate='one_to_one')

result = pd.DataFrame({
    'Kiểm tra': ['Revenue', 'COGS'],
    'Số ngày đối chiếu': [len(check), len(check)],
    'Sai số tuyệt đối lớn nhất': [
        (check.Revenue-check.Revenue_rebuilt).abs().max(),
        (check.COGS-check.COGS_rebuilt).abs().max()],
    'Số ngày lệch quá 0.01': [
        ((check.Revenue-check.Revenue_rebuilt).abs() > .01).sum(),
        ((check.COGS-check.COGS_rebuilt).abs() > .01).sum()]
})
display(result)
display(check[['Date','Revenue','Revenue_rebuilt','COGS','COGS_rebuilt']].head())

,Kiểm tra,Số ngày đối chiếu,Sai số tuyệt đối lớn nhất,Số ngày lệch quá 0.01
0,Revenue,3833,1.862645e-09,0
1,COGS,3833,4.999269e-03,0


,Date,Revenue,Revenue_rebuilt,COGS,COGS_rebuilt
0,2012-07-04,5123547.94,5123547.94,3982991.19,3.982991e+06
1,2012-07-05,2751773.45,2751773.45,2150580.23,2.150580e+06
2,2012-07-06,3054029.42,3054029.42,2517632.84,2.517633e+06
3,2012-07-07,2667930.94,2667930.94,2108246.62,2.108247e+06
4,2012-07-08,2360851.90,2360851.90,1808622.79,1.808623e+06


## Kết luận trình bày trong 30 giây

> “Thiết kế không bắt đầu từ việc muốn có 19 bảng. Em bắt đầu từ grain và một giao dịch thật, kiểm tra khóa, phụ thuộc hàm và các cột có thể suy ra. Khóa dòng hàng sai đã tạo fan-out và lỗi nghiệp vụ giả; các cột city/zip bị lưu lặp; promotion đang là nhóm lặp; sales tái tạo được từ giao dịch. Từ các bằng chứng đó em mới đưa ra schema. Riêng `region` là lựa chọn master data, không phải yêu cầu bắt buộc của 3NF.”

Nếu giảng viên hỏi bất kỳ file nào, quay về ba câu:

1. Một dòng đại diện cho cái gì?
2. Khóa của dòng đó là gì?
3. Nó gắn vào đâu trong câu chuyện bán hàng?